In [1]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
from math import sqrt
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from matplotlib.patches import FancyArrowPatch,FancyBboxPatch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import xlsxwriter



warnings.filterwarnings('ignore')

In [2]:

possible_dirs = [".", "data", "./data", "../data"]
INPUT_DIR = "."

for dir_path in possible_dirs:
    if os.path.exists(dir_path):
        csv_files = glob.glob(os.path.join(dir_path, "*_Data.csv"))
        if csv_files and len(csv_files) >= 5:
            INPUT_DIR = dir_path
            break

print(f"✓ CSV fajlovi pronađeni u: {INPUT_DIR}")
print(f"✓ Sve biblioteke učitane!")

✓ CSV fajlovi pronađeni u: data
✓ Sve biblioteke učitane!


In [3]:

## 📋 FAZA 1: Priprema podataka
def load_csv(path):
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [c.strip().strip('"') for c in df.columns]
    return df


def clean_numeric_column(series):
    """Konvertuje string kolonu (zarezi/procenti/K,M,B sufiksi) u numeričku."""
    s = series.astype(str).str.replace(",", "", regex=False)
    s = s.str.replace("%", "", regex=False)

    def parse_suffix(val):
        if not isinstance(val, str):
            return np.nan if pd.isna(val) else float(val)
        val = val.strip()
        if val in ("", "nan", "None", "-"):
            return np.nan
        multiplier = 1
        if val.endswith("K"):
            multiplier, val = 1_000, val[:-1]
        elif val.endswith("M"):
            multiplier, val = 1_000_000, val[:-1]
        elif val.endswith("B"):
            multiplier, val = 1_000_000_000, val[:-1]
        try:
            return float(val) * multiplier
        except ValueError:
            return np.nan

    return s.apply(parse_suffix)


csv_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*_Data.csv")))
print(f"Pronađeno {len(csv_files)} CSV fajlova:\n")

all_data = []
missing_report = []

for path in csv_files:
    pair_name = os.path.basename(path).replace("_Data.csv", "")
    df_raw = load_csv(path)

    df_raw["Date"] = pd.to_datetime(df_raw["Date"], format="%m/%d/%Y", errors="coerce")

    numeric_cols = [c for c in ["Price", "Open", "High", "Low", "Change %"] if c in df_raw.columns]
    for col in numeric_cols:
        df_raw[col] = clean_numeric_column(df_raw[col])

    total_rows = len(df_raw)
    cols_to_check = ["Date"] + numeric_cols
    missing_counts = df_raw[cols_to_check].isna().sum()
    missing_pct = (missing_counts / total_rows * 100).round(2)

    report_row = {"Pair": pair_name, "Total_Rows": total_rows}
    for col in cols_to_check:
        report_row[f"{col}_missing_%"] = missing_pct[col]
    missing_report.append(report_row)

    keep_cols = ["Date"] + numeric_cols
    df = df_raw[keep_cols].dropna(subset=["Date", "Price"])
    df["Pair"] = pair_name

    all_data.append(df)
    print(f"  ✓ {pair_name:10s}: {len(df)} redova (od {total_rows} ukupno)")

# --- Ukloni redove iz USDCNY čiji datum se ne javlja kod NIJEDNOG drugog para ---
other_dates = set()
for df in all_data:
    if df["Pair"].iloc[0] != "USDCNY":
        other_dates.update(df["Date"])

for i, df in enumerate(all_data):
    if df["Pair"].iloc[0] == "USDCNY":
        before = len(df)
        all_data[i] = df[df["Date"].isin(other_dates)]
        removed = before - len(all_data[i])
        if removed > 0:
            print(f"\n  ⚠ USDCNY: uklonjeno {removed} redova sa datumima koji se ne javljaju kod ostalih parova "
                  f"({before} → {len(all_data[i])})")

currency_all = pd.concat(all_data, ignore_index=True)
currency_all = currency_all.sort_values(["Pair", "Date"]).reset_index(drop=True)

print(f"\n✓ Ukupno: {len(currency_all)} redova za {currency_all['Pair'].nunique()} parova")

Pronađeno 6 CSV fajlova:

  ✓ EURUSD    : 2610 redova (od 2610 ukupno)
  ✓ USDCAD    : 2610 redova (od 2610 ukupno)
  ✓ USDCHF    : 2610 redova (od 2610 ukupno)
  ✓ USDCNY    : 2611 redova (od 2611 ukupno)
  ✓ USDGBP    : 2610 redova (od 2610 ukupno)
  ✓ USDJPY    : 2610 redova (od 2610 ukupno)

  ⚠ USDCNY: uklonjeno 1 redova sa datumima koji se ne javljaju kod ostalih parova (2611 → 2610)

✓ Ukupno: 15660 redova za 6 parova


In [4]:
missing_df = pd.DataFrame(missing_report)
print("="*80)
print("UDEO NEDOSTAJUĆIH VREDNOSTI PO PARU (%)")
print("="*80 + "\n")
print(missing_df.to_string(index=False))

UDEO NEDOSTAJUĆIH VREDNOSTI PO PARU (%)

  Pair  Total_Rows  Date_missing_%  Price_missing_%  Open_missing_%  High_missing_%  Low_missing_%  Change %_missing_%
EURUSD        2610             0.0              0.0             0.0             0.0            0.0                 0.0
USDCAD        2610             0.0              0.0             0.0             0.0            0.0                 0.0
USDCHF        2610             0.0              0.0             0.0             0.0            0.0                 0.0
USDCNY        2611             0.0              0.0             0.0             0.0            0.0                 0.0
USDGBP        2610             0.0              0.0             0.0             0.0            0.0                 0.0
USDJPY        2610             0.0              0.0             0.0             0.0            0.0                 0.0


In [5]:
from ta.trend import SMAIndicator, EMAIndicator, MACD, CCIIndicator, PSARIndicator
from ta.momentum import ROCIndicator, RSIIndicator
from ta.volatility import AverageTrueRange
from stockstats import StockDataFrame

INDICATOR_COLS = [
    "SMA_14", "EMA_14", "ROC_14", "RSI_14",
    "MACD", "MACD_Signal", "MACD_Hist",
    "STD_14", "ATR_14",
    "KAMA_14", "SAR", "CCI_14", "CMO_14"
]

def calculate_indicators(pair_df):
    close = pair_df["Price"]
    high = pair_df["High"] if "High" in pair_df.columns else close
    low = pair_df["Low"] if "Low" in pair_df.columns else close

    out = pd.DataFrame(index=pair_df.index)

    out["SMA_14"] = SMAIndicator(close=close, window=14).sma_indicator()
    out["EMA_14"] = EMAIndicator(close=close, window=14).ema_indicator()
    out["ROC_14"] = ROCIndicator(close=close, window=14).roc()
    out["RSI_14"] = RSIIndicator(close=close, window=14).rsi()

    macd = MACD(close=close, window_slow=26, window_fast=12, window_sign=9)
    out["MACD"]        = macd.macd()
    out["MACD_Signal"] = macd.macd_signal()
    out["MACD_Hist"]   = macd.macd_diff()

    out["STD_14"] = close.rolling(14).std()
    out["ATR_14"] = AverageTrueRange(high=high, low=low, close=close, window=14).average_true_range()
    out["CCI_14"] = CCIIndicator(high=high, low=low, close=close, window=14).cci()

    psar = PSARIndicator(high=high, low=low, close=close, step=0.025, max_step=0.25)
    out["SAR"] = psar.psar()

    stock = StockDataFrame.retype(pd.DataFrame({
        "close": close.values,
        "high": high.values,
        "low": low.values,
        "open": close.values,
        "volume": np.ones(len(close))
    }, index=pair_df.index))

    out["KAMA_14"] = stock["kama_14"]
    out["CMO_14"]  = stock["cmo_14"]

    return out



print("Računanje indikatora pomoću `ta` paketa...\n")

all_features = []
for pair_name in currency_all["Pair"].unique():
    pair_data = currency_all[currency_all["Pair"] == pair_name].sort_values("Date").reset_index(drop=True)
    indicators = calculate_indicators(pair_data)
    indicators["Date"] = pair_data["Date"].values
    indicators["Price"] = pair_data["Price"].values
    indicators["Pair"] = pair_name
    all_features.append(indicators)
    print(f"  ✓ {pair_name}: {len(indicators)} redova")

features = pd.concat(all_features, ignore_index=True)
features = features[["Pair", "Date", "Price"] + INDICATOR_COLS]

print(f"\n✓ Shape: {features.shape}")

  ✓ USDJPY: 2610 redova

✓ Shape: (15660, 16)


In [6]:
def add_roc(df, cols, n):
    out = pd.DataFrame(index=df.index)
    for c in cols:
        roc = df[c].pct_change(n) * 100
        roc = roc.replace([np.inf, -np.inf], np.nan)
        out[f"ROC{n}_{c}"] = roc
    return out

def add_sma(df, cols, n):
    out = pd.DataFrame(index=df.index)
    for c in cols:
        out[f"SMA{n}_{c}"] = df[c].rolling(n).mean()
    return out

def add_price_targets(df, horizons):
    out = pd.DataFrame(index=df.index)
    for h in horizons:
        out[f"Target_t+{h}"] = df["Price"].shift(-h)
    return out

print("✓ Helper funkcije definisane (ROC, SMA, target generatori)")

✓ Helper funkcije definisane (ROC, SMA, target generatori)


In [7]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

N_LAGS = 5
N_LAGS_INDICATORS = 2
TEST_SIZE = 0.2

MLP_PARAMS = {
    'hidden_layer_sizes': (128, 64, 32),
    'activation': 'relu',
    'solver': 'adam',
    'learning_rate_init': 0.001,
    'max_iter': 2000,
    'early_stopping': True,
    'validation_fraction': 0.1,
    'n_iter_no_change': 25,
    'alpha': 0.01,
    'random_state': 42,
    'verbose': 0,
}

def mape_safe(y_true, y_pred, eps=1e-9):
    mask = np.abs(y_true) > eps
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def direction_accuracy(y_true, y_pred):
    delta_true = np.diff(y_true)
    delta_pred = np.diff(y_pred)
    if len(delta_true) == 0:
        return np.nan
    correct = np.sign(delta_true) == np.sign(delta_pred)
    return correct.mean() * 100  # %

def plot_and_save_results(y_test, y_pred, title, filename):
    plt.figure(figsize=(12, 6))
    plt.plot(y_test, label='Stvarna vrednost', color='blue', alpha=0.6, linewidth=2)
    plt.plot(y_pred, label='Predikcija modela', color='red', linestyle='--', alpha=0.8)
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.savefig(filename)
    plt.close()



def save_confusion_matrix(y_test, y_pred, title, filename):
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pad', 'Rast'], yticklabels=['Pad', 'Rast'])
    plt.title(title)
    plt.ylabel('Stvarno')
    plt.xlabel('Predviđeno')
    plt.savefig(filename)
    plt.close()

def run_regression(X, y):
    # Split
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]

    # Skaliranje
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    model = MLPRegressor(**MLP_PARAMS)
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)

    # Metrike
    y_test, y_pred = y_test.ravel(), y_pred.ravel()

    # Pretvaranje u smer za klasifikacione metrike
    y_test_dir = (np.diff(y_test, prepend=y_test[0]) > 0).astype(int)
    y_pred_dir = (np.diff(y_pred, prepend=y_pred[0]) > 0).astype(int)

    cm = confusion_matrix(y_test_dir, y_pred_dir).ravel()

    return {
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred),
        "MAPE": mape_safe(y_test, y_pred),
        "DirAcc": direction_accuracy(y_test, y_pred),
        "Accuracy": accuracy_score(y_test_dir, y_pred_dir),
        "Precision": precision_score(y_test_dir, y_pred_dir, zero_division=0),
        "Recall": recall_score(y_test_dir, y_pred_dir, zero_division=0),
        "F1": f1_score(y_test_dir, y_pred_dir, zero_division=0),
        "TN": cm[0], "FP": cm[1], "FN": cm[2], "TP": cm[3],
        "y_test": y_test, "y_pred": y_pred
    }

In [8]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def add_return_targets(df, horizons):
    out = pd.DataFrame(index=df.index)
    for h in horizons:
        out[f"Target_ROC_t+{h}"] = (df["Price"].shift(-h) - df["Price"]) / df["Price"] * 100
    return out

def add_trend_targets(df, horizons):
    out = pd.DataFrame(index=df.index)
    for h in horizons:
        future_ret = df["Price"].shift(-h) - df["Price"]
        out[f"Target_SignROC_t+{h}"] = np.where(future_ret > 0, 1, 0)
        out.loc[future_ret.isna(), f"Target_SignROC_t+{h}"] = np.nan
    return out

def run_classification(X, y):
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    model = MLPClassifier(**MLP_PARAMS)
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)

    cm = confusion_matrix(y_test, y_pred).ravel()

    return {
        # MAE/RMSE/R2/MAPE/DirAcc nemaju smisla za klasifikaciju -> NaN, ne 0,
        # da se ne mešaju sa stvarnim regresionim nulama u izveštaju.
        "MAE": np.nan, "RMSE": np.nan, "R2": np.nan, "MAPE": np.nan, "DirAcc": np.nan,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "TN": cm[0], "FP": cm[1], "FN": cm[2], "TP": cm[3],
        "y_test": y_test, "y_pred": y_pred
    }

print("✓ Helper funkcije za prinose i klasifikaciju definisane.")

✓ Helper funkcije za prinose i klasifikaciju definisane.


In [9]:
def get_features_for_exp(exp_id, price_lags, indicator_lags, roc_price, sma_price, roc_ind, sma_ind):
    if exp_id == '1-1':
        return {
            'a) cene': (price_lags, 'standard'),
            'b) indikatori': (indicator_lags, 'robust'),
            'c) cene+indikatori': (pd.concat([price_lags, indicator_lags], axis=1), 'robust')
        }
    elif exp_id == '1-2':
        return {
            'a) cene+ROC(cena)': (pd.concat([price_lags, roc_price], axis=1), 'standard'),
            'b) indikatori+ROC(ind)': (pd.concat([indicator_lags, roc_ind], axis=1), 'robust'),
            'c) cene+ind+ROC(oba)': (pd.concat([price_lags, roc_price, indicator_lags, roc_ind], axis=1), 'robust')
        }
    elif exp_id == '1-3':
        return {
            'a) cene+ROC+SMA(cena)': (pd.concat([price_lags, roc_price, sma_price], axis=1), 'standard'),
            'b) ind+ROC+SMA(ind)': (pd.concat([indicator_lags, roc_ind, sma_ind], axis=1), 'robust'),
            'c) sve kombinovano': (pd.concat([price_lags, roc_price, sma_price, indicator_lags, roc_ind, sma_ind], axis=1), 'robust')
        }

# --- GLAVNA PETLJA ---
results_all = []
if not os.path.exists('CM_Plots'): os.makedirs('CM_Plots')

experiments = {
    '1-1': {'type': 'reg', 'horizons': [1, 5, 10]},
    '1-2': {'type': 'reg', 'horizons': [1, 3, 5, 10]},
    '1-3': {'type': 'reg', 'horizons': [1, 3, 5, 10]},
    '2-1': {'type': 'reg', 'horizons': [1, 3, 5, 10]},
    '2-2': {'type': 'reg', 'horizons': [1, 3, 5, 10]},
    '2-3': {'type': 'reg', 'horizons': [1, 3, 5, 10]},
    '3-1': {'type': 'cls', 'horizons': [1, 3, 5, 10]},
    '3-2': {'type': 'cls', 'horizons': [1, 3, 5, 10]},
    '3-3': {'type': 'cls', 'horizons': [1, 3, 5, 10]} # Sada je cls
}
detailed_preds = []

# Sve kolone metrika koje pamtimo iz run_regression/run_classification rezultata
METRIC_KEYS = ['MAE', 'RMSE', 'R2', 'MAPE', 'DirAcc',
               'Accuracy', 'Precision', 'Recall', 'F1', 'TN', 'FP', 'FN', 'TP']

for pair_name in features['Pair'].unique():
    pdf = features[features['Pair'] == pair_name].sort_values('Date').reset_index(drop=True)

    price_lags = pd.DataFrame({f'Price_lag{i}': pdf['Price'].shift(i) for i in range(1, N_LAGS + 1)}, index=pdf.index)
    ind_lags = pd.DataFrame({f'{c}_lag{i}': pdf[c].shift(i) for c in INDICATOR_COLS for i in range(1, 3)}, index=pdf.index)

    for exp_id in experiments.keys():
        if exp_id == '1': targets = add_price_targets(pdf, experiments[exp_id]['horizons'])
        elif exp_id == '2': targets = add_return_targets(pdf, experiments[exp_id]['horizons'])
        else: targets = add_trend_targets(pdf, experiments[exp_id]['horizons'])

        for sub in ['a', 'b', 'c']:
            if sub == 'a': X_df = price_lags
            elif sub == 'b': X_df = ind_lags
            else: X_df = pd.concat([price_lags, ind_lags], axis=1)

            for h in experiments[exp_id]['horizons']:
                col_name = f'Target_t+{h}' if exp_id=='1' else (f'Target_ROC_t+{h}' if exp_id=='2' else f'Target_SignROC_t+{h}')
                combo = pd.concat([X_df, targets[[col_name]], pdf[['Date']]], axis=1).dropna()

                if len(combo) < 50: continue

                if experiments[exp_id]['type'] == 'reg':
                    metrics = run_regression(combo[X_df.columns].values, combo[col_name].values)
                else:
                    metrics = run_classification(combo[X_df.columns].values, combo[col_name].values)
                    cm = confusion_matrix(metrics['y_test'], metrics['y_pred']).ravel()
                    metrics.update({'TN': cm[0], 'FP': cm[1], 'FN': cm[2], 'TP': cm[3]})

                row = {'Exp': f'{exp_id}-{sub}', 'Pair': pair_name, 'H': h}
                row.update({k: metrics.get(k, np.nan) for k in METRIC_KEYS})
                results_all.append(row)

                for actual, pred in zip(metrics['y_test'], metrics['y_pred']):
                    detailed_preds.append({
                        'Exp': f'{exp_id}-{sub}',
                        'Pair': pair_name,
                        'H': h,
                        'Stvarna_Vrednost': actual,
                        'Predikcija': pred,
                        'Greska': abs(actual - pred)
                    })

In [10]:

df_final = pd.DataFrame(results_all)
df_detailed = pd.DataFrame(detailed_preds)

def export_results(df_final, df_detailed, base_name):
    # 1. Fajl sa metrikom (Svi_Rezultati_Metrike.xlsx)
    metrics_file = f"{base_name}_Metrike.xlsx"
    writer_metrics = pd.ExcelWriter(metrics_file, engine='xlsxwriter')

    # Sada uključujemo i MAPE i DirAcc (Direction Accuracy)
    all_cols = ['Exp', 'Pair', 'H', 'MAE', 'RMSE', 'R2', 'MAPE', 'DirAcc',
                'Accuracy', 'Precision', 'Recall', 'F1', 'TN', 'FP', 'FN', 'TP']
    df_full = df_final.reindex(columns=all_cols)
    df_full.to_excel(writer_metrics, sheet_name='Svi_Rezultati', index=False)

    workbook = writer_metrics.book
    header_format = workbook.add_format({'bold': True, 'fg_color': '#2F75B5', 'font_color': 'white', 'border': 1, 'align': 'center'})
    ws = writer_metrics.sheets['Svi_Rezultati']
    ws.freeze_panes(1, 0)
    for col_num, col_name in enumerate(df_full.columns):
        ws.write(0, col_num, col_name, header_format)
        ws.set_column(col_num, col_num, 12)
    writer_metrics.close()

    # 2. Fajl sa detaljnim predikcijama (Svi_Rezultati_Predikcije.xlsx)
    preds_file = f"{base_name}_Predikcije.xlsx"
    writer_preds = pd.ExcelWriter(preds_file, engine='xlsxwriter')
    df_detailed.to_excel(writer_preds, sheet_name='Pojedinacne_Predikcije', index=False)

    workbook2 = writer_preds.book
    header_format2 = workbook2.add_format({'bold': True, 'fg_color': '#2F75B5', 'font_color': 'white', 'border': 1, 'align': 'center'})
    ws_p = writer_preds.sheets['Pojedinacne_Predikcije']
    ws_p.freeze_panes(1, 0)
    num_format = workbook2.add_format({'num_format': '0.0000'})
    for col_num, col_name in enumerate(df_detailed.columns):
        ws_p.write(0, col_num, col_name, header_format2)
        ws_p.set_column(col_num, col_num, 15, num_format)

    writer_preds.close()
    print(f"✓ Kreirana dva fajla: '{metrics_file}' i '{preds_file}'.")

# POZIV:
export_results(df_final, df_detailed, 'Analiza_Modeli')

✓ Kreirana dva fajla: 'Analiza_Modeli_Metrike.xlsx' i 'Analiza_Modeli_Predikcije.xlsx'.
